# hvad-skal-vi-seerr - training notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shuresm57/hvad-skal-vi-seerr/blob/main/train.ipynb)

Trains a tiny matrix-factorization movie recommender on the [MovieLens 100k](https://grouplens.org/datasets/movielens/100k/) dataset.

Run every cell top to bottom (`Runtime > Run all`). The last cell downloads `models.zip`.
Unzip it into the `models/` folder of your local clone of this repo, then run locally:

```
pip install -r requirements-infer.txt
python recommend.py --user_id 1
```

In [ ]:
!pip install -q -r https://raw.githubusercontent.com/shuresm57/hvad-skal-vi-seerr/main/requirements-train.txt

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -q -o ml-100k.zip

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
N_FACTORS = 32
EPOCHS = 20
BATCH_SIZE = 1024
LR = 0.01
WEIGHT_DECAY = 1e-4

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp']
)
movies = pd.read_csv(
    'ml-100k/u.item', sep='|', encoding='latin-1', header=None, usecols=[0, 1],
    names=['item_id', 'title'],
)

user_ids = sorted(ratings['user_id'].unique())
item_ids = sorted(ratings['item_id'].unique())
user_id_map = {str(uid): idx for idx, uid in enumerate(user_ids)}
item_id_map = {str(iid): idx for idx, iid in enumerate(item_ids)}

ratings['user_idx'] = ratings['user_id'].map(lambda x: user_id_map[str(x)])
ratings['item_idx'] = ratings['item_id'].map(lambda x: item_id_map[str(x)])

perm = rng.permutation(len(ratings))
split = int(0.9 * len(ratings))
train_ratings = ratings.iloc[perm[:split]].reset_index(drop=True)
test_ratings = ratings.iloc[perm[split:]].reset_index(drop=True)

n_users, n_items = len(user_ids), len(item_ids)
print(f'{n_users} users, {n_items} items, {len(train_ratings)} train / {len(test_ratings)} test ratings')

In [ ]:
import torch.nn as nn


class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, n_factors):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))
        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.item_emb.weight, std=0.05)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx):
        dot = (self.user_emb(user_idx) * self.item_emb(item_idx)).sum(1)
        bias = self.user_bias(user_idx).squeeze(1) + self.item_bias(item_idx).squeeze(1)
        return self.global_bias + bias + dot


model = MatrixFactorization(n_users, n_items, N_FACTORS)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()

In [ ]:
train_user = torch.tensor(train_ratings['user_idx'].values, dtype=torch.long)
train_item = torch.tensor(train_ratings['item_idx'].values, dtype=torch.long)
train_rating = torch.tensor(train_ratings['rating'].values, dtype=torch.float32)

n_train = len(train_user)

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    for start in range(0, n_train, BATCH_SIZE):
        batch_idx = perm[start:start + BATCH_SIZE]
        u, i, r = train_user[batch_idx], train_item[batch_idx], train_rating[batch_idx]

        optimizer.zero_grad()
        pred = model(u, i)
        loss = loss_fn(pred, r)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(batch_idx)

    if epoch % 5 == 0 or epoch == 1:
        print(f'epoch {epoch:>2}  train_mse={epoch_loss / n_train:.4f}')

In [ ]:
model.eval()
test_user = torch.tensor(test_ratings['user_idx'].values, dtype=torch.long)
test_item = torch.tensor(test_ratings['item_idx'].values, dtype=torch.long)
test_rating = torch.tensor(test_ratings['rating'].values, dtype=torch.float32)

with torch.no_grad():
    test_pred = model(test_user, test_item)
    test_rmse = torch.sqrt(loss_fn(test_pred, test_rating)).item()

print(f'test RMSE: {test_rmse:.4f}')

In [ ]:
models_dir = Path('models')
models_dir.mkdir(exist_ok=True)

np.save(models_dir / 'user_emb.npy', model.user_emb.weight.detach().numpy())
np.save(models_dir / 'item_emb.npy', model.item_emb.weight.detach().numpy())
np.save(models_dir / 'user_bias.npy', model.user_bias.weight.detach().numpy().squeeze(1))
np.save(models_dir / 'item_bias.npy', model.item_bias.weight.detach().numpy().squeeze(1))

(models_dir / 'global_bias.json').write_text(
    json.dumps({'global_bias': model.global_bias.item()})
)
(models_dir / 'user_id_map.json').write_text(json.dumps(user_id_map))
(models_dir / 'item_id_map.json').write_text(json.dumps(item_id_map))

seen_items = (
    ratings.groupby('user_id')['item_id']
    .apply(lambda ids: [str(i) for i in ids])
    .to_dict()
)
seen_items = {str(uid): items for uid, items in seen_items.items()}
(models_dir / 'seen_items.json').write_text(json.dumps(seen_items))

movies.to_csv(models_dir / 'movies.csv', index=False)

(models_dir / 'metrics.json').write_text(json.dumps({
    'test_rmse': test_rmse,
    'n_factors': N_FACTORS,
    'epochs': EPOCHS,
    'n_users': n_users,
    'n_items': n_items,
    'n_ratings': len(ratings),
}, indent=2))

print('Artifacts written to', models_dir.resolve())
print(sorted(p.name for p in models_dir.iterdir()))

In [ ]:
import shutil

shutil.make_archive('models', 'zip', root_dir='.', base_dir='models')

try:
    from google.colab import files
    files.download('models.zip')
except ImportError:
    print('Not running in Colab - models.zip was created in the current directory, copy it manually.')

## What gets exported

- `user_emb.npy`, `item_emb.npy`, `user_bias.npy`, `item_bias.npy` - model weights
- `global_bias.json`, `user_id_map.json`, `item_id_map.json`, `seen_items.json` - lookup tables
- `movies.csv` - item_id -> title
- `metrics.json` - test RMSE and training config, so you have a record of what you actually trained before trusting the artifact